In [42]:
# we are now going to try 4-6 more proteins to see if the ESM-2-8M, Layer 4, Feature 5712 association with the amino acid Proline holds
# we will avoid picking close relatives of glcophorin A as they provide less independent evidence
# we are looking for possible exceptions (not really obtaining a representative estimate across all proteins)

In [43]:
# we can have a look at some residue contact maps as well?

In [44]:
# pseudo idea
# membrane vs soluble across proteins with clustered prolines, scattered prolines, no/few prolines
# this forms a 2 x 3 punnet square

Biological context for me:

A cell membrane is the phospholipid bilayer, with the hydrophobic interior (fatty acid chains/tails) and the hydrophilic exterior (the head which consists of a phosphate group and an alcohol residue). The membrane distinction (y-axis of table below) describes how a protein sits in its biological environment. We want to see if feature 5712's Proline association persists across different biological settings.

**Integral Membrane Protein** - Part of the protein is embedded in the lipid bilayer.

**Transmembrane Protein** - An *integral membrane protein* whose chain crosses the bilayer (multi-pass means it crosses several times).

**Soluble Protein** - Can exist in the watery environment inside or outside cells without requiring insertion into a membrane.

**Peripheral Membrane Protein** - Associates with the membrane surface or with another membrane protein, without being embedded like an integral protein.

Descriptions are from:

https://www.rcsb.org/docs/general-help/membrane-protein-resources

https://www.ncbi.nlm.nih.gov/books/NBK26878/

#### Development collection: testing the proline-associated hypothesis

Proteins selected before inspecting their feature activations.

The aim is to challenge the interpretation of ESM-2-8M layer-4 SAE feature 5712 across different protein and sequence contexts.

| Protein class | Clustered prolines | Scattered prolines | No/few prolines |
|---|---|---|---|
| **Integral membrane** | **T-cell surface antigen CD2** — human; **P06729**; 351 residues. A single-pass membrane protein with proline-rich regions in its cytoplasmic tail. | **Bacteriorhodopsin** — *Halobacterium salinarum*; **P02945**; 262 residues. A multi-pass membrane protein with dispersed prolines, including prolines within membrane-spanning helices. | **Phospholamban** — human; **P26678**; 52 residues. A single-pass membrane protein with very few prolines, providing many non-proline positions to examine. |
| **Soluble / no membrane-spanning segment** | **Vasodilator-stimulated phosphoprotein (VASP)** — human; **P50552**; 380 residues. An actin-associated protein with a central proline-rich region. | **Carbonic anhydrase II** — human; **P00918**; 260 residues. A folded enzyme with prolines distributed across the sequence rather than concentrated in a long proline-rich region. | **Calmodulin-1** — human; **P0DP23**; 149 residues. A calcium-binding protein with very few prolines, offering a different structural context from phospholamban. |

These are development proteins, not held-out evaluation proteins.
Close **homologues** of each should remain in the same group in any later split.
Note homologues are proteins related by common evolutionary ancestry; close homologues often have very similar sequences.

For example, human CD2 in development and a very similar CD2 from another species in evaluation could make the evaluation too easy. Therefore, we keep both in the development group to leave evaluation for more independent examples.

“Clustered”, “scattered”, and “few” are provisional descriptions.
We will record proline counts and positions from the canonical sequences
before examining activations. Short local pairs can occur in a protein
classified as having scattered prolines.

Lengths refer to full canonical sequences.

In [45]:
import requests # needed to download UniProt records or structure files
from pprint import pprint  # Displays nested data in a readable layout.

accession = "P06729"
url = f"https://rest.uniprot.org/uniprotkb/{accession}.json"

response = requests.get(url, timeout=30)
response.raise_for_status()  # Stop if UniProt returns an HTTP error.

record = response.json()  # Convert the JSON response into Python data.
print(f"record is a python {type(record)}")

print("Top-level fields:")
pprint(list(record.keys()))

print("\nSequence section:")
pprint(record["sequence"]) # this is also a dict

record is a python <class 'dict'>
Top-level fields:
['entryType',
 'primaryAccession',
 'secondaryAccessions',
 'uniProtkbId',
 'entryAudit',
 'annotationScore',
 'organism',
 'proteinExistence',
 'proteinDescription',
 'genes',
 'comments',
 'features',
 'keywords',
 'references',
 'uniProtKBCrossReferences',
 'sequence',
 'extraAttributes']

Sequence section:
{'crc64': 'A03D853C3B618917',
 'length': 351,
 'md5': 'FC09A16399B6A30723D437A22FB64AA7',
 'molWeight': 39448,
 'value': 'MSFPCKFVASFLLIFNVSSKGAVSKEITNALETWGALGQDINLDIPSFQMSDDIDDIKWEKTSDKKKIAQFRKEKETFKEKDTYKLFKNGTLKIKHLKTDDQDIYKVSIYDTKGKNVLEKIFDLKIQERVSKPKISWTCINTTLTCEVMNGTDPELNLYQDGKHLKLSQRVITHKWTTSLSAKFKCTAGNKVSKESSVEPVSCPEKGLDIYLIIGICGGGSLLMVFVALLVFYITKRKKQRSRRNDEELETRAHRVATEERGRKPHQIPASTPQNPATSQHPPPPPGHRSQAPSHRPPPPGHRVQHQPQKRPPAPSGTQVHQQKGPPLPRPRVQPKPPHGAAENSLSPSSN'}


UniProt’s `molWeight` gives the calculated molecular mass in daltons (Da). A protein of 40,000 Da has a molar mass of approximately 40,000 g/mol; Avogadro’s constant connects the mass of one molecule to the mass of a mole of molecules. (throwback to school)

In [46]:
def get_amino_acid_sequence(proteins: list[str]):
    """Fetch canonical sequences for a non-empty list of UniProt accession strings.
    
    Return a dictionary mapping accessions to amino-acid sequences.
    """

    # Check every entry before making any requests to UniProt.
    if not isinstance(proteins, list):
        raise TypeError("Please supply a list of UniProt accession strings.")
    if not proteins:
        raise ValueError("Please supply at least one UniProt accession.")
    if not all(isinstance(protein, str) for protein in proteins):
        raise TypeError("Every UniProt accession must be a string.")
    if any(not protein.strip() for protein in proteins):
        raise ValueError("UniProt accessions must not be empty or blank.")

    amino_acid_dict = {}

    for protein in proteins:
        accession = protein
        url = f"https://rest.uniprot.org/uniprotkb/{accession}.json"
    
        response = requests.get(url, timeout=30)
        response.raise_for_status()  # Stop if UniProt returns an HTTP error.
        record = response.json()  # Convert the JSON response into Python data.v

        if protein in amino_acid_dict:
            pass
        else:
            amino_acid_dict[f'{protein}'] = record['sequence']['value']

    return amino_acid_dict

In [47]:
amino_acid_dict = get_amino_acid_sequence([
    'P06729',  # CD2
    'P02945',  # Bacteriorhodopsin
    'P26678',  # Phospholamban
    'P50552',  # VASP
    'P00918',  # Carbonic anhydrase II
    'P0DP23',  # Calmodulin-1
])

In [48]:
amino_acid_dict

{'P06729': 'MSFPCKFVASFLLIFNVSSKGAVSKEITNALETWGALGQDINLDIPSFQMSDDIDDIKWEKTSDKKKIAQFRKEKETFKEKDTYKLFKNGTLKIKHLKTDDQDIYKVSIYDTKGKNVLEKIFDLKIQERVSKPKISWTCINTTLTCEVMNGTDPELNLYQDGKHLKLSQRVITHKWTTSLSAKFKCTAGNKVSKESSVEPVSCPEKGLDIYLIIGICGGGSLLMVFVALLVFYITKRKKQRSRRNDEELETRAHRVATEERGRKPHQIPASTPQNPATSQHPPPPPGHRSQAPSHRPPPPGHRVQHQPQKRPPAPSGTQVHQQKGPPLPRPRVQPKPPHGAAENSLSPSSN',
 'P02945': 'MLELLPTAVEGVSQAQITGRPEWIWLALGTALMGLGTLYFLVKGMGVSDPDAKKFYAITTLVPAIAFTMYLSMLLGYGLTMVPFGGEQNPIYWARYADWLFTTPLLLLDLALLVDADQGTILALVGADGIMIGTGLVGALTKVYSYRFVWWAISTAAMLYILYVLFFGFTSKAESMRPEVASTFKVLRNVTVVLWSAYPVVWLIGSEGAGIVPLNIETLLFMVLDVSAKVGFGLILLRSRAIFGEAEAPEPSAGDGAAATSD',
 'P26678': 'MEKVQYLTRSAIRRASTIEMPQQARQKLQNLFINFCLILICLLLICIIVMLL',
 'P50552': 'MSETVICSSRATVMLYDDGNKRWLPAGTGPQAFSRVQIYHNPTANSFRVVGRKMQPDQQVVINCAIVRGVKYNQATPNFHQWRDARQVWGLNFGSKEDAAQFAAGMASALEALEGGGPPPPPALPTWSVPNGPSPEEVEQQKRQQPGPSEHIERRVSNAGGPPAPPAGGPPPPPGPPPPPGPPPPPGLPPSGVPAAAHGAGGGPPPAPPLPAAQGPGGGGAGAPGLAAAIAGAKLRKVSKQEEASGGPTAPKAESGRSGGGGLMEEMNAMLARRRKAT

In [49]:
# Give each sequence a name using its UniProt accession.
P06729_sequence = amino_acid_dict['P06729']
P02945_sequence = amino_acid_dict['P02945']
P26678_sequence = amino_acid_dict['P26678']
P50552_sequence = amino_acid_dict['P50552']
P00918_sequence = amino_acid_dict['P00918']
P0DP23_sequence = amino_acid_dict['P0DP23']

In [50]:
import os
from pathlib import Path

# Tell InterPLM where to keep its project data.
# Set this before importing its inference functions.
os.environ["INTERPLM_DATA"] = str(Path("data/interplm").resolve())

import numpy as np
import matplotlib.pyplot as plt
import torch

from interplm.embedders import ESM
from interplm.sae.inference import load_sae_from_hf

/opt/anaconda3/envs/interpreting-plms/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [52]:
# start with the small, 8-million-parameter model and layer 4 - this layer is an exploratory starting choice
layer = 4

esm_model = ESM(
    model_name="facebook/esm2_t6_8M_UR50D", # publisher / model name (facebook is the publisher’s organisation name on HF)
    device="cpu",
)

In [ ]:
# Run every sequence through the same loaded model at layer 4.
embeddings_by_protein = {
    accession: esm_model.embed_single_sequence(sequence, layer=layer)
    for accession, sequence in amino_acid_dict.items() # items method returns key value pairs
}

# rows vary with sequence length (the amino acid chains are different lengths)
# all arrays should have 320 columns(embeddings are 320 dimensional vectors)
for accession, embeddings in embeddings_by_protein.items():
    assert isinstance(embeddings, np.ndarray), f"{accession}: expected a NumPy array."
    expected_shape = (len(amino_acid_dict[accession]), 320)
    assert embeddings.shape == expected_shape, (
        f"{accession}: expected shape {expected_shape}, got {embeddings.shape}."
    )

P06729_embeddings = embeddings_by_protein['P06729']
P02945_embeddings = embeddings_by_protein['P02945']
P26678_embeddings = embeddings_by_protein['P26678']
P50552_embeddings = embeddings_by_protein['P50552']
P00918_embeddings = embeddings_by_protein['P00918']
P0DP23_embeddings = embeddings_by_protein['P0DP23']

# store the shapes of the embeddings of the different proteins/accesions in a dict (for printing purposes)
embedding_shapes = {
    accession: embeddings.shape
    for accession, embeddings in embeddings_by_protein.items()
}

print(f"All embeddings are {np.ndarray}.\nShapes (residues, dimensions): {embedding_shapes}")

All embeddings are <class 'numpy.ndarray'>.
Shapes (residues, dimensions): {'P06729': (351, 320), 'P02945': (262, 320), 'P26678': (52, 320), 'P50552': (380, 320), 'P00918': (260, 320), 'P0DP23': (149, 320)}


In [63]:
# load the pretrained sparse autoencoder (sae) - this is an InterPLM autoencoder
# the autoencoder must match the ESM-2 model and layer used above
sae = load_sae_from_hf(
    plm_model="esm2-8m",
    plm_layer=layer,
)
sae = sae.to("cpu").eval() # switch the sae to evaluation mode - apply its learned parameters

Loading configs from /Users/ishanmitra/.cache/huggingface/hub/models--Elana--InterPLM-esm2-8m/snapshots/81d2429cd9dae7175f1dcd8b4c649a20cdc06c8c/layer_4/config.yaml
Loaded data type: <class 'interplm.train.configs.TrainingRunConfig'>
Data keys: Not a dict


In [64]:
# Reuse the loaded SAE to calculate every feature for each protein.
activations_by_protein = {}
with torch.inference_mode():
    for accession, embeddings in embeddings_by_protein.items():
        activations_by_protein[accession] = sae.encode_feat_subset(
            torch.as_tensor(embeddings, dtype=torch.float32),
            list(range(sae.dict_size)),
            normalize_features=True,
        ).cpu().numpy()

# One row per residue and one column per SAE feature.
for accession, activations in activations_by_protein.items():
    assert isinstance(activations, np.ndarray), f"{accession}: expected a NumPy array."
    expected_shape = (len(amino_acid_dict[accession]), sae.dict_size)
    assert activations.shape == expected_shape, (
        f"{accession}: expected shape {expected_shape}, got {activations.shape}."
    )

P06729_activations = activations_by_protein['P06729']
P02945_activations = activations_by_protein['P02945']
P26678_activations = activations_by_protein['P26678']
P50552_activations = activations_by_protein['P50552']
P00918_activations = activations_by_protein['P00918']
P0DP23_activations = activations_by_protein['P0DP23']

activation_shapes = {
    accession: activations.shape
    for accession, activations in activations_by_protein.items()
}
print(f"All activations are {np.ndarray}.\nShapes (residues, features): {activation_shapes}")

All activations are <class 'numpy.ndarray'>.
Shapes (residues, features): {'P06729': (351, 10240), 'P02945': (262, 10240), 'P26678': (52, 10240), 'P50552': (380, 10240), 'P00918': (260, 10240), 'P0DP23': (149, 10240)}
